In [69]:
import os
import json
import pandas as pd
from openai import OpenAI ## Suggest version:1.12.0
import time
os.environ["OPENAI_API_KEY"] = "Your OpenAI Key here"

## RAG with OpenAI Assistant API

In [ ]:
org_game = """
Based on the knowledge in the paper, answer the following question:
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request?

Tell me the number and the reason in the following example format, and nothing else:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [51]:
client_rag = OpenAI()
file = client_rag.files.create(
    file=open("RAG_data/Arad-1120MoneyRequest-2012(OCR).pdf", 'rb'),
    purpose="assistants"
)

In [52]:
assistant_gpt4 = client_rag.beta.assistants.create(
    name = "11-20 game expert",
    tools = [{"type":"retrieval"}],
    model = "gpt-4-1106-preview",
    file_ids = [file.id],
)

In [82]:
assistant_gpt3 = client_rag.beta.assistants.create(
    name = "11-20 game expert",
    tools = [{"type":"retrieval"}],
    model = "gpt-3.5-turbo-0125", 
    file_ids = [file.id],
)

In [ ]:
lst_3 = []
for i in range(1000):
    print(i)
    thread = client_rag.beta.threads.create()
    print(thread)
    
    message = client_rag.beta.threads.messages.create(
    thread_id = thread.id,
    role = "user",
    content = org_game #prompt here
    )
    run = client_rag.beta.threads.runs.create(
    thread_id = thread.id,
    assistant_id= assistant_gpt3.id
    )

    while True:
    # Retrieve the run status
        time.sleep(10)
        run_status = client_rag.beta.threads.runs.retrieve(thread_id=thread.id,run_id=run.id)
        if run_status.status == 'completed':
            print("successfully load")
            
            messages = client_rag.beta.threads.messages.list(thread_id=thread.id)
            lst_3.append(messages.data[0].content[0].text.value)
            break
        else:
            ### sleep again
            print("connection error, try again..")

In [ ]:
result = []
for i in range(len(lst_3)):
    try:
        result.append(json.loads(lst_3[i]))
    except json.JSONDecodeError:
        print(i)
        continue
print(len(result))

In [ ]:
dt = pd.DataFrame(result)
from collections import Counter
print(Counter(dt[:1000]['number']))

## Finetuning

In [ ]:
org = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? Answer the number you choose and reason in a json format.
"""

In [95]:
client_finetune = OpenAI()
client_finetune.files.create(
  file=open("finetune_data/human.jsonl", "rb"),
  purpose="fine-tune"
)

FileObject(id='file-hfPq1Xu5gJJuiurt1XKDa4dB', bytes=71670, created_at=1728873444, filename='human.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)

In [ ]:
client_finetune.fine_tuning.jobs.create(
  training_file="Your file id", 
  model="gpt-3.5-turbo-0125"
)

In [ ]:
client_finetune.fine_tuning.jobs.create(
  training_file="Your file id", 
  model="gpt-4o-2024-08-06"
)

In [ ]:
client_finetune.fine_tuning.jobs.list(limit=3) #Check Finetune process

In [102]:
def gpt3_finetune(prompt):
    client = OpenAI()
    completion = client.chat.completions.create(
        model="Your own finetuned model name here",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return completion.choices[0].message.content


In [103]:
def gpt4_finetune(prompt):
    client = OpenAI()
    completion = client.chat.completions.create(
        model="Your own finetuned model name here",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
lst3_finetune = []
for i in range(1000):
    lst3_finetune.append(gpt3_finetune(org_game))

In [ ]:
lst4_finetune = []
for i in range(1000):
    lst4_finetune.append(gpt4_finetune(org_game))

In [ ]:
result = []
for i in range(len(lst3_finetune)):
    try:
        result.append(json.loads(lst3_finetune[i]))
    except json.JSONDecodeError:
        print(i)
        continue
print(len(result))

In [ ]:
dt = pd.DataFrame(result)
from collections import Counter
print(Counter(dt[:1000]['number']))